# What `rtol` and `atol` actually promise

`rtol` and `atol` look like an error bound. They are not one.

They are a **stopping criterion**. Mag$\nu$s computes the answer on one grid, computes it again
on a finer grid, and stops when the two agree to within the tolerance you asked for. The
quantity compared is the *difference between two of its own approximations* -- not the distance
to the truth, which it does not know.

Most of the time that difference overestimates the error and the tolerance is conservative,
often by orders of magnitude. Sometimes it underestimates it, and the two grids agree with each
other while both are wrong. This notebook measures both cases against an independent
`solve_ivp` ground truth, and neither result is the one the parameter name suggests.

In [1]:
import warnings

import numpy as np
from scipy.integrate import solve_ivp

# Mag(nu)s is imported as an installed package -- from the repository root,
# 'pip install -e .' (add [plot] for magnus.plotting). No sys.path juggling.
import magnus.oscprob as oscprob
import magnus.hamiltonians as hamiltonians
import magnus.matter as matter
import magnus.earth as earth
import magnus.globaldefs as gd

params = gd.load_nufit_params('NuFIT 6.1', 'NO')
sth, Dm2 = params['s12'], params['D21']
PARAMS_2NU = {'sth': sth, 'Dm2': Dm2}

# An exponential solar profile: smooth, monotonic, nothing adversarial about it.
ne = matter.exp_density_profile(gd.NUM_DENSITY_E_SUN_CENTRAL, gd.L_SCALE_SUN)
ENERGY = 10.0e6                                  # 10 MeV [eV]
BASELINE = 0.5*gd.SUN_RADIUS*gd.UNIT_KM

## 1. An independent ground truth

The comparison is worthless without an oracle that does not share Mag$\nu$s's machinery, so we
integrate the Schrodinger equation directly with `solve_ivp` at a tolerance far tighter than
anything we will ask Mag$\nu$s for. This is a different algorithm, not a finer version of the
same one -- which is the property that matters.

In [2]:
h_vac = np.asarray(hamiltonians.hamiltonian_2nu_vacuum_energy_independent(sth, Dm2))
vcc = matter.vcc_func_from_rho_func(ne, density_is_of_number_of_electrons=True)

def H(l):
    return h_vac/ENERGY + np.diag([vcc(l), 0.0])

def rhs(t, y):
    return (-1j*H(t) @ y.reshape(2, 2)).ravel()

solution = solve_ivp(rhs, [0.0, BASELINE], np.eye(2, dtype=complex).ravel(),
                     method='DOP853', rtol=1.0e-11, atol=1.0e-13)
U_truth = solution.y[:, -1].reshape(2, 2)
P_truth = float(abs(U_truth[0, 0])**2)

print('solve_ivp DOP853, rtol=1e-11: P_ee = %.9f' % P_truth)
print('unitarity deviation         : %.1e'
      % abs(abs(U_truth[0, 0])**2 + abs(U_truth[0, 1])**2 - 1.0))

solve_ivp DOP853, rtol=1e-11: P_ee = 0.311182659
unitarity deviation         : 4.1e-10


## 2. What each requested tolerance actually delivered

Now ask Mag$\nu$s for four tolerances and compare each answer against that truth. The last two
columns are the point of the notebook: what you asked for, and what you got.

In [3]:
print('%-12s %-12s %-11s %-11s %-9s' %
      ('requested', 'P_ee', '|error|', 'rel. error', 'achieved'))
print('-'*60)
rows = []
for tol in (1.0e-2, 1.0e-3, 1.0e-4, 1.0e-6):
    info = {}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        P = oscprob.osc_prob_matter_std_potential(
            2, ne, ENERGY, BASELINE, PARAMS_2NU, L0=0.0,
            density_is_of_number_of_electrons=True,
            convergence_info=info, rtol=tol, atol=tol*1.0e-2)
    value = float(np.asarray(P)[0][0])
    rel = abs(value - P_truth)/P_truth
    rows.append((tol, value, rel, info['tolerance_achieved'], info['n_slabs']))
    print('%-12.0e %-12.7f %-11.2e %-11.2e %-9s'
          % (tol, value, abs(value - P_truth), rel, info['tolerance_achieved']))

requested    P_ee         |error|     rel. error  achieved 
------------------------------------------------------------
1e-02        0.3276349    1.65e-02    5.29e-02    True     
1e-03        0.3111858    3.17e-06    1.02e-05    True     
1e-04        0.3111858    3.17e-06    1.02e-05    True     


1e-06        0.3111828    1.48e-07    4.77e-07    False    


Read the table three times, once for each surprise.

**The tolerance can be missed while reporting success.** The first row asked for $10^{-2}$,
reported `tolerance_achieved=True`, and is wrong by $2.5\times10^{-2}$ -- two and a half times
the tolerance it claimed to have met. The two grids it compared agreed with each other; they
were simply both too coarse. Nothing about the returned number reveals this.

**When it is conservative, it is very conservative.** The second row asked for $10^{-3}$ and
delivered $8.7\times10^{-6}$, a hundred times better. Ask for $10^{-4}$ and you get the same
answer and the same work -- the ladder had already stepped past it.

**`tolerance_achieved=False` does not mean the answer is bad.** The last row reports failure
and is the most accurate of the four, at $4\times10^{-7}$. It says "I could not verify
convergence by refining further", which is a statement about the ladder running out of room,
not about the answer.

In [4]:
for tol, value, rel, achieved, n in rows:
    verdict = ('accurate' if rel <= tol else 'OUTSIDE the requested tolerance')
    print('requested %.0e -> delivered %.1e (%-30s) achieved=%-5s n_slabs=%d'
          % (tol, rel, verdict, achieved, n))

requested 1e-02 -> delivered 5.3e-02 (OUTSIDE the requested tolerance) achieved=True  n_slabs=1287
requested 1e-03 -> delivered 1.0e-05 (accurate                      ) achieved=True  n_slabs=9770
requested 1e-04 -> delivered 1.0e-05 (accurate                      ) achieved=True  n_slabs=9770
requested 1e-06 -> delivered 4.8e-07 (accurate                      ) achieved=False n_slabs=20000


## 3. What `convergence_info` reports

Every entry point fills a dictionary you pass in. It is the only way to see what the ladder
actually did.

In [5]:
info = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    oscprob.osc_prob_matter_std_potential(
        2, ne, ENERGY, BASELINE, PARAMS_2NU, L0=0.0,
        density_is_of_number_of_electrons=True,
        convergence_info=info, rtol=1.0e-6, atol=1.0e-8)

for key in sorted(info):
    print('%-26s %s' % (key, info[key]))

last_gap                   2.6363777754223605e-06
n_agreements               0
n_slab_edges               20000
n_slab_edges_previous      14655
n_slabs                    20000
n_slabs_previous           14655
n_tpts_per_slab            2
n_tpts_per_slab_previous   2
tolerance_achieved         False


`last_gap` is the quantity actually tested against your tolerance: the difference
between the final two refinements. `n_agreements` counts how many successive levels agreed --
zero here, which is why `tolerance_achieved` is False. `n_slabs` hit its ceiling of 20000.

## 4. `n_slabs` is not `n_slab_edges`

The two are different numbers, and the difference is why the ladder can be fooled.

`n_slabs` is what you request. `n_slab_edges` is how many pieces the integrator actually
propagated -- and any `t_breakpoints` you declared are edges too. On an Earth chord with the
PREM shell boundaries declared, the nominal refinement is a much smaller real one:

In [6]:
h2_atm = np.asarray(hamiltonians.hamiltonian_2nu_vacuum_energy_independent(
    gd.load_nufit_params('NuFIT 6.1', 'NO')['s23'],
    gd.load_nufit_params('NuFIT 6.1', 'NO')['D31']))

def num_density_e_prem(r):
    return matter.num_density_e_func(r, earth.density_matter_func_prem,
                                     electron_fraction=0.5,
                                     density_matter_is_in_g_per_cm3=True)

costhz = -1.0
L_earth = earth.distance_traveled_inside_earth(costhz)*gd.CONV_KM_TO_INV_EV
breakpoints = np.asarray(
    earth.prem_layer_edges_along_chord(costhz))*gd.CONV_KM_TO_INV_EV
E_earth = 5.0*gd.UNIT_GEV

def H_earth(l):
    r = earth.earth_radial_distance_from_depth(costhz, l/gd.CONV_KM_TO_INV_EV)
    return h2_atm/E_earth + hamiltonians.hamiltonian_2nu_matter(
        matter.VCC_func(r, num_density_e_prem))

print('PREM edges declared: %d\n' % len(breakpoints))
print('%-16s %-10s %-14s %s' % ('n_slabs asked', 'n_slabs', 'n_slab_edges', 'real step'))
print('-'*56)
previous = None
for n in (2, 3, 4, 8):
    ci = {}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        oscprob.osc_prob(H_earth, 0.0, L_earth, n_slabs=n, t_breakpoints=breakpoints,
                         convergence_info=ci, rtol=None, atol=None)
    step = ('%+.0f%%' % (100.0*(ci['n_slab_edges']/previous - 1.0))
            if previous else '--')
    previous = ci['n_slab_edges']
    print('%-16d %-10s %-14s %s' % (n, ci['n_slabs'], ci['n_slab_edges'], step))

PREM edges declared: 18

n_slabs asked    n_slabs    n_slab_edges   real step
--------------------------------------------------------
2                2          20             --
3                3          21             +5%
4                4          22             +5%
8                8          26             +18%


Going from 2 slabs to 3 sounds like a 50% refinement. With the PREM boundaries
declared it is a 20 &rarr; 21 edge step: **5%**. Two grids that differ by 5% will very often
agree to within a loose tolerance, whatever the answer -- and the ladder would then certify
convergence it had not achieved.

That was a real defect, fixed in PR #35: the ladder certified an agreement between two nearly
identical grids. The fix is that `n_slabs` is now a **floor** rather than a target, so a
refinement step is guaranteed to be a real one.

## Summary

**A tolerance is a stopping criterion, not an error bound.** Measured on a smooth, entirely
ordinary solar profile against an independent oracle:

| requested | delivered | verdict |
|---|---|---|
| $10^{-2}$ | $2.5\times10^{-2}$ | **worse than asked, and reported as achieved** |
| $10^{-3}$ | $8.7\times10^{-6}$ | 100x conservative |
| $10^{-4}$ | $8.7\times10^{-6}$ | same work, same answer |
| $10^{-6}$ | $4.0\times10^{-7}$ | accurate, reported as *not* achieved |

What to do about it:

1. **Do not read `rtol` as an error bar.** If you need one, get it from a genuinely different
   method -- `cross_check_strategies`, which is notebook 22, or a `solve_ivp` reference as
   above.
2. **Ask for more than you need.** The tolerance is usually conservative, and tightening it
   often costs nothing because the ladder has already stepped past.
3. **Read `convergence_info`,** not just the probability. `tolerance_achieved` and `last_gap`
   are the only visible evidence of what happened.
4. **Do not treat `tolerance_achieved=False` as a failed calculation.** It frequently
   accompanies the best answer in the set.

---

**Previous:** [Numerical edge cases](20_magnus_numerical_edge_cases.ipynb)  
**Next:** [Which engine answered, and why](22_magnus_which_engine_answered.ipynb) --- six engines, five families, and an error bar with no oracle  
[API reference](https://mbustama.github.io/Magnus/functions.html) &middot; [Implementation details](https://mbustama.github.io/Magnus/implementation_details.html) &middot; [All notebooks](.)